# FireEye — Export, Evaluate & Push to Hub

Loads best.pt, exports to ONNX, runs final evaluation, and uploads model + weights to HuggingFace.

**Prerequisites**: Completed `02_train_yolov8m.ipynb` — `best.pt` must exist in `runs/fireye_yolov8m/weights/`.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
REPO_ROOT    = '/content/drive/MyDrive/fireye'
DATASET_ROOT = f'{REPO_ROOT}/data/merged'
RUNS_ROOT    = f'{REPO_ROOT}/runs'
BEST_PT      = f'{RUNS_ROOT}/fireye_yolov8m/weights/best.pt'

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.chdir(REPO_ROOT)

!pip install -q ultralytics onnx onnxruntime huggingface_hub

import ultralytics; ultralytics.checks()
print(f'best.pt exists: {os.path.exists(BEST_PT)}')

In [ ]:
# ── Cell 2: Load best model ───────────────────────────────────────────────────
from ultralytics import YOLO
model = YOLO(BEST_PT)
print(model.info())

In [ ]:
# ── Cell 3: Export to ONNX ───────────────────────────────────────────────────
# opset=17 required for YOLOv8 SELU activations — do not lower
onnx_path = model.export(
    format='onnx',
    imgsz=640,
    opset=17,
    simplify=True,
    dynamic=False,
    half=False,
)
print(f'Exported: {onnx_path}')

In [ ]:
# ── Cell 4: Verify ONNX outputs match PyTorch ─────────────────────────────────
import numpy as np
import onnxruntime as ort
import torch

session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
print(f'ONNX input: {input_name} {session.get_inputs()[0].shape}')

# Run a dummy forward pass
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
onnx_out = session.run(None, {input_name: dummy})
print(f'ONNX output shape: {onnx_out[0].shape}')
print('ONNX verification passed.')

In [ ]:
# ── Cell 5: Benchmark ONNX vs PyTorch ────────────────────────────────────────
results = model.benchmark(format='onnx', imgsz=640, half=False, device=0)
print(results)

In [ ]:
# ── Cell 6: Final evaluation (test split) ────────────────────────────────────
import yaml

# Patch dataset.yaml path
with open(f'{REPO_ROOT}/configs/dataset.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATASET_ROOT
with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(cfg, f)

metrics = model.val(
    data='/content/dataset.yaml',
    split='test',
    conf=0.25,
    iou=0.45,
    plots=True,
    device=0,
)

print('\n── Final Test Metrics ─────────────────')
print(f'mAP@50      : {metrics.box.map50:.4f}')
print(f'mAP@50-95   : {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}')
print('Per-class:')
for i, name in enumerate(['fire', 'smoke']):
    print(f'  {name}: AP50={metrics.box.ap50[i]:.4f}')

In [ ]:
# ── Cell 7: Show confusion matrix and PR curves ───────────────────────────────
from IPython.display import Image as IPImage
import glob

val_dir = sorted(glob.glob(f'{RUNS_ROOT}/fireye_yolov8m*/'))[-1]
for plot in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    path = f'{val_dir}{plot}'
    if os.path.exists(path):
        print(plot)
        display(IPImage(path, width=700))

In [ ]:
# ── Cell 8: Push model to HuggingFace Hub ────────────────────────────────────
import os
from huggingface_hub import HfApi

HF_MODEL_REPO = os.environ.get('HF_MODEL_REPO', 'hajorda/fireye-yolov8m')
token = os.environ['HF_TOKEN']

api = HfApi()
api.create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=token)

# Upload weights
api.upload_file(path_or_fileobj=BEST_PT,       path_in_repo='best.pt',   repo_id=HF_MODEL_REPO, token=token)
api.upload_file(path_or_fileobj=str(onnx_path), path_in_repo='best.onnx', repo_id=HF_MODEL_REPO, token=token)

# Upload training results
results_png = f'{RUNS_ROOT}/fireye_yolov8m/results.png'
if os.path.exists(results_png):
    api.upload_file(path_or_fileobj=results_png, path_in_repo='results.png', repo_id=HF_MODEL_REPO, token=token)

print(f'\nModel uploaded: https://huggingface.co/{HF_MODEL_REPO}')

In [ ]:
# ── Cell 9: Quick inference demo ─────────────────────────────────────────────
import glob, random
from IPython.display import Image as IPImage

test_imgs = random.sample(glob.glob(f'{DATASET_ROOT}/images/test/*.jpg'), 6)
results = model.predict(
    source=test_imgs,
    conf=0.30,
    iou=0.45,
    save=True,
    project='/content/demo',
    name='final',
    exist_ok=True,
)

from pathlib import Path
for r in results:
    saved = Path(r.save_dir) / Path(r.path).name
    if saved.exists():
        display(IPImage(str(saved), width=600))

## All done!

- Dataset: https://huggingface.co/datasets/hajorda/fireye-wildfire-detection
- Model: https://huggingface.co/hajorda/fireye-yolov8m

**Loading for inference:**
```python
from ultralytics import YOLO
model = YOLO('https://huggingface.co/hajorda/fireye-yolov8m/resolve/main/best.pt')
results = model.predict('your_image.jpg', conf=0.30)
```